In [ ]:
%pip install --quiet PyMuPDF langchain-text-splitters requests

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./fsr_config.py

In [ ]:
import json, time, hashlib
from pathlib import Path
from datetime import datetime, timezone

import fitz  # PyMuPDF
import requests
import urllib3
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType, TimestampType,
)

urllib3.disable_warnings()
spark = SparkSession.builder.getOrCreate()

# ── Test overrides — REMOVE BEFORE PRODUCTION ───────────────────────────────
METADATA_TABLE = f"{POC_CATALOG}.{POC_SCHEMA}.fsr_metadata_registry_test"
CHUNK_TABLE    = f"{POC_CATALOG}.{POC_SCHEMA}.fsr_chunks_test"
VS_INDEX_NAME  = f"{POC_CATALOG}.{POC_SCHEMA}.vs_fsr_chunks_test"
LITELLM_API_KEY = "sk-cjRRha3Ejczz8AmnJKyQxA"

P2_MAX_PDFS = 10  # cap for testing

print("=== Process 2 — Chunk Ingestion ===")
print(f"  Metadata table : {METADATA_TABLE}")
print(f"  Chunk table    : {CHUNK_TABLE}")
print(f"  VS index       : {VS_INDEX_NAME}")
print(f"  VS endpoint    : {VS_ENDPOINT_NAME}")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Embed dimension: {EMBEDDING_DIMENSION}")
print(f"  Chunk size     : {CHUNKING_CONFIG.chunk_size}")
print(f"  Chunk overlap  : {CHUNKING_CONFIG.chunk_overlap}")
print(f"  LLM base URL   : {LITELLM_BASE_URL}")
print(f"  API key set    : {bool(LITELLM_API_KEY)}")
print(f"  Max PDFs       : {P2_MAX_PDFS}")

In [ ]:
# ── Create chunk table if it doesn't exist ──────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
        {CHUNK_TABLE_DDL_COLS}
    )
    USING DELTA
    COMMENT 'FSR chunk rows with materialized metadata — TEST'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[OK] Chunk table ready: {CHUNK_TABLE}")

# ── Query metadata table for documents ready to chunk ───────────────────────
# Only process docs where metadata extraction succeeded (completed) and chunking is
# pending or previously failed.
pending_df = spark.sql(f"""
    SELECT pdf_name, volume_path, title, esn, equipment_type,
           event_type, report_issued_date, page_count
    FROM {METADATA_TABLE}
    WHERE metadata_status = '{MetadataStatus.COMPLETED}'
      AND chunk_status IN ('{ChunkStatus.PENDING}', '{ChunkStatus.FAILED}')
""")
pending_docs = pending_df.collect()

if P2_MAX_PDFS and len(pending_docs) > P2_MAX_PDFS:
    pending_docs = pending_docs[:P2_MAX_PDFS]

print(f"[OK] {len(pending_docs)} documents to chunk")
for row in pending_docs[:5]:
    print(f"  {row.pdf_name[:50]}  pages={row.page_count}")
if len(pending_docs) > 5:
    print(f"  ... and {len(pending_docs) - 5} more")

In [ ]:
# ── Extract full text from each PDF using PyMuPDF ───────────────────────────
# Returns: {pdf_name: {pages: [{page_num, text}], total_pages, error}}

doc_texts = {}   # pdf_name -> {pages: [...], total_pages: int}
doc_errors = {}  # pdf_name -> error_msg

for row in pending_docs:
    doc_key = row.pdf_name
    vol_path = row.volume_path
    try:
        doc = fitz.open(vol_path)
        total_pages = len(doc)
        pages = []
        for page_idx in range(total_pages):
            text = doc[page_idx].get_text()
            if text and text.strip():
                pages.append({
                    "page_num": page_idx + 1,  # 1-based
                    "text": text,
                })
        doc.close()

        if not pages:
            doc_errors[doc_key] = "No extractable text in PDF"
            print(f"  [SKIP] {row.pdf_name[:50]}: no text")
            continue

        doc_texts[doc_key] = {
            "pages": pages,
            "total_pages": total_pages,
        }
        total_chars = sum(len(p["text"]) for p in pages)
        print(f"  [OK] {row.pdf_name[:50]}  pages={len(pages)}/{row.page_count}  chars={total_chars:,}")

    except Exception as e:
        doc_errors[doc_key] = str(e)[:500]
        print(f"  [FAIL] {row.pdf_name[:50]}: {e}")

print(f"\n[OK] Extracted: {len(doc_texts)}, Failed: {len(doc_errors)}")

In [ ]:
# ── Recursive chunking with page tracking ───────────────────────────────────
# We join all page texts and track page boundaries so each chunk knows its
# start_page and end_page.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNKING_CONFIG.chunk_size,
    chunk_overlap=CHUNKING_CONFIG.chunk_overlap,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

doc_chunks = {}  # pdf_name -> [{chunk_index, chunk_text, start_page, end_page, chunk_size}]

for doc_key, text_data in doc_texts.items():
    pages = text_data["pages"]

    # Build full text and page offset map
    full_text = ""
    page_spans = []  # [(start_offset, end_offset, page_num)]
    for page in pages:
        start = len(full_text)
        full_text += page["text"] + "\n\n"
        end = len(full_text)
        page_spans.append((start, end, page["page_num"]))

    if not full_text.strip():
        doc_errors[doc_key] = "Empty text after joining pages"
        continue

    # Split into chunks
    chunk_texts = splitter.split_text(full_text)

    chunks = []
    search_from = 0
    for ci, ctext in enumerate(chunk_texts):
        # Locate chunk in full_text to determine page range
        idx = full_text.find(ctext, search_from)
        if idx == -1:
            idx = full_text.find(ctext.strip(), max(0, search_from - 200))
        if idx >= 0:
            c_start = idx
            c_end = idx + len(ctext)
            chunk_pages = [pn for (ps, pe, pn) in page_spans
                           if ps < c_end and pe > c_start]
            search_from = idx + 1
        else:
            chunk_pages = [p["page_num"] for p in pages]

        start_page = min(chunk_pages) if chunk_pages else 1
        end_page = max(chunk_pages) if chunk_pages else text_data["total_pages"]

        chunks.append({
            "chunk_index": ci,
            "chunk_text": ctext,
            "start_page": start_page,
            "end_page": end_page,
            "chunk_size": len(ctext),
        })

    doc_chunks[doc_key] = chunks
    print(f"  [OK] {doc_key[:50]}  chunks={len(chunks)}  "
          f"avg_size={sum(c['chunk_size'] for c in chunks) // max(len(chunks), 1)}")

print(f"\n[OK] Chunked {len(doc_chunks)} documents, "
      f"total chunks: {sum(len(c) for c in doc_chunks.values())}")

In [ ]:
# ── Generate embeddings via LiteLLM ────────────────────────────────────────
# Batch all chunk texts, call the embedding endpoint, store vectors.

EMBED_BATCH_SIZE = 32  # texts per API call
EMBED_MAX_RETRIES = 3

def embed_batch(texts):
    """Call LiteLLM embedding endpoint. Returns list of float vectors."""
    base = LITELLM_BASE_URL.rstrip("/")
    urls = [f"{base}/v1/embeddings", f"{base}/embeddings"]
    payload = {
        "model": EMBEDDING_MODEL,
        "input": texts,
    }
    headers = {
        "Authorization": f"Bearer {LITELLM_API_KEY}",
        "Content-Type": "application/json",
    }
    for attempt in range(1, EMBED_MAX_RETRIES + 1):
        for url in urls:
            try:
                resp = requests.post(url, headers=headers, json=payload,
                                     timeout=120, verify=LLM_VERIFY_SSL)
                if resp.status_code == 404:
                    continue
                resp.raise_for_status()
                data = resp.json().get("data", [])
                if len(data) != len(texts):
                    raise RuntimeError(
                        f"Expected {len(texts)} embeddings, got {len(data)}")
                ordered = sorted(data, key=lambda x: x.get("index", 0))
                return [item["embedding"] for item in ordered]
            except Exception as e:
                if "CERTIFICATE_VERIFY_FAILED" in str(e):
                    continue
                if attempt == EMBED_MAX_RETRIES:
                    raise
                wait = 5 * (2 ** (attempt - 1))
                print(f"    [WARN] Embed attempt {attempt} failed ({e}); retrying in {wait}s")
                time.sleep(wait)
    raise RuntimeError("Embedding call failed after all retries")


# Flatten all chunks for batched embedding
all_chunk_refs = []  # (pdf_name, chunk_index)
all_chunk_texts = []

for doc_key, chunks in doc_chunks.items():
    for chunk in chunks:
        all_chunk_refs.append((doc_key, chunk["chunk_index"]))
        all_chunk_texts.append(chunk["chunk_text"])

print(f"Generating embeddings for {len(all_chunk_texts)} chunks "
      f"(batch size={EMBED_BATCH_SIZE})...")

all_embeddings = {}  # (pdf_name, chunk_index) -> [float, ...]
t0 = time.time()

for i in range(0, len(all_chunk_texts), EMBED_BATCH_SIZE):
    batch_texts = all_chunk_texts[i:i + EMBED_BATCH_SIZE]
    batch_refs = all_chunk_refs[i:i + EMBED_BATCH_SIZE]
    batch_num = (i // EMBED_BATCH_SIZE) + 1
    total_batches = (len(all_chunk_texts) + EMBED_BATCH_SIZE - 1) // EMBED_BATCH_SIZE

    vectors = embed_batch(batch_texts)
    for ref, vec in zip(batch_refs, vectors):
        all_embeddings[ref] = vec

    print(f"  Batch {batch_num}/{total_batches}: {len(batch_texts)} texts  "
          f"dim={len(vectors[0])}  elapsed={time.time() - t0:.1f}s")

print(f"\n[OK] Generated {len(all_embeddings)} embeddings in {time.time() - t0:.1f}s")

# Verify dimension
if all_embeddings:
    sample_dim = len(next(iter(all_embeddings.values())))
    print(f"  Embedding dimension: {sample_dim} (expected {EMBEDDING_DIMENSION})")
    if sample_dim != EMBEDDING_DIMENSION:
        print(f"  [WARN] Dimension mismatch! Update EMBEDDING_DIMENSION in config.")

In [ ]:
# ── Write chunk rows to Delta + update metadata table ───────────────────────

# Build a lookup from pdf_name -> metadata row
doc_metadata = {}
for row in pending_docs:
    doc_metadata[row.pdf_name] = row

# Assemble all chunk rows
chunk_rows = []
for doc_key, chunks in doc_chunks.items():
    meta = doc_metadata.get(doc_key)
    if not meta:
        continue
    chunk_count = len(chunks)
    for chunk in chunks:
        ci = chunk["chunk_index"]
        chunk_id = hashlib.md5(
            f"{doc_key}_{ci}".encode("utf-8")
        ).hexdigest()
        embedding = all_embeddings.get((doc_key, ci))
        if embedding is None:
            continue

        chunk_rows.append({
            "chunk_id": chunk_id,
            "pdf_name": doc_key,
            "chunk_index": ci,
            "chunk_text": chunk["chunk_text"],
            "embedding": [float(v) for v in embedding],
            "title": meta.title,
            "esn": meta.esn,
            "equipment_type": meta.equipment_type,
            "event_type": meta.event_type,
            "report_issued_date": meta.report_issued_date,
            "page_count": meta.page_count,
            "chunk_count": chunk_count,
            "start_page": chunk["start_page"],
            "end_page": chunk["end_page"],
            "chunk_size": chunk["chunk_size"],
        })

print(f"[OK] Assembled {len(chunk_rows)} chunk rows for {len(doc_chunks)} documents")

# ── Write to Delta ──────────────────────────────────────────────────────────
if chunk_rows:
    schema = StructType([
        StructField("chunk_id", StringType(), False),
        StructField("pdf_name", StringType(), False),
        StructField("chunk_index", IntegerType(), False),
        StructField("chunk_text", StringType(), False),
        StructField("embedding", ArrayType(DoubleType()), True),
        StructField("title", StringType(), True),
        StructField("esn", StringType(), True),
        StructField("equipment_type", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("report_issued_date", StringType(), True),
        StructField("page_count", IntegerType(), True),
        StructField("chunk_count", IntegerType(), True),
        StructField("start_page", IntegerType(), True),
        StructField("end_page", IntegerType(), True),
        StructField("chunk_size", IntegerType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])

    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    for row in chunk_rows:
        row["ingested_at"] = now

    chunk_df = spark.createDataFrame(chunk_rows, schema=schema)
    chunk_df.createOrReplaceTempView("_fsr_new_chunks")

    # MERGE: insert new chunks, update existing (re-chunked docs)
    spark.sql(f"""
        MERGE INTO {CHUNK_TABLE} AS tgt
        USING _fsr_new_chunks AS src
        ON tgt.chunk_id = src.chunk_id
        WHEN MATCHED THEN UPDATE SET
            tgt.chunk_text = src.chunk_text,
            tgt.embedding = src.embedding,
            tgt.title = src.title,
            tgt.esn = src.esn,
            tgt.equipment_type = src.equipment_type,
            tgt.event_type = src.event_type,
            tgt.report_issued_date = src.report_issued_date,
            tgt.source_volume = src.source_volume,
            tgt.page_count = src.page_count,
            tgt.page_count = src.page_count,
            tgt.chunk_count = src.chunk_count,
            tgt.start_page = src.start_page,
            tgt.end_page = src.end_page,
            tgt.chunk_size = src.chunk_size,
            tgt.ingested_at = src.ingested_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"[OK] MERGE complete: {len(chunk_rows)} chunk rows")

    # ── Update metadata table: chunk_status → completed ─────────────────────
    success_doc_ids = list(doc_chunks.keys())
    success_pdf_names = list(doc_chunks.keys())
    if success_pdf_names:
        ids_sql = ", ".join(f"'{pn}'" for pn in success_pdf_names)
            UPDATE {METADATA_TABLE}
            SET chunk_status = '{ChunkStatus.COMPLETED}',
                chunk_error = NULL,
                updated_at = current_timestamp()
            WHERE document_id IN ({ids_sql})
            WHERE pdf_name IN ({ids_sql})
        print(f"[OK] Updated {len(success_doc_ids)} docs → chunk_status=completed")
        print(f"[OK] Updated {len(success_pdf_names)} docs → chunk_status=completed")
# ── Handle failures: mark chunk_status=failed ──────────────────────────────
if doc_errors:
    for doc_id, err in doc_errors.items():
    for doc_key, err in doc_errors.items():
        spark.sql(f"""
            UPDATE {METADATA_TABLE}
            SET chunk_status = '{ChunkStatus.FAILED}',
                chunk_error = '{safe_err}',
                updated_at = current_timestamp()
            WHERE document_id = '{doc_id}'
            WHERE pdf_name = '{doc_key}'
    print(f"[OK] Marked {len(doc_errors)} docs → chunk_status=failed")

# ── Verify ──────────────────────────────────────────────────────────────────
chunk_count = spark.sql(f"SELECT COUNT(*) AS n FROM {CHUNK_TABLE}").first().n
doc_count = spark.sql(f"SELECT COUNT(DISTINCT document_id) AS n FROM {CHUNK_TABLE}").first().n
doc_count = spark.sql(f"SELECT COUNT(DISTINCT pdf_name) AS n FROM {CHUNK_TABLE}").first().n
display(spark.sql(f"""
    SELECT document_id, COUNT(*) AS chunks, MIN(chunk_size) AS min_size,
    SELECT pdf_name, COUNT(*) AS chunks, MIN(chunk_size) AS min_size,
    FROM {CHUNK_TABLE}
    GROUP BY document_id
    GROUP BY pdf_name

In [ ]:
# ── Trigger Vector Search index sync ────────────────────────────────────────
# This cell creates or syncs the Delta Sync index backed by the chunk table.
# For the test run, the VS endpoint may not be available — this cell is
# best-effort and will print warnings rather than fail the notebook.

import os

def get_dbr_auth():
    ws_url = "https://gevernova-ai-dev-dbr.cloud.databricks.com"
    token = None
    try:
        token = (dbutils.notebook.entry_point
                 .getDbutils().notebook().getContext()
                 .apiToken().get())
    except Exception:
        token = os.getenv("DATABRICKS_TOKEN", "")
    return ws_url, token or ""


def sync_vector_search():
    ws_url, token = get_dbr_auth()
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    # Check endpoint status
    ep_url = f"{ws_url}/api/2.0/vector-search/endpoints/{VS_ENDPOINT_NAME}"
    try:
        resp = requests.get(ep_url, headers=headers, timeout=30, verify=False)
        if resp.ok:
            state = resp.json().get("endpoint_status", {}).get("state", "")
            print(f"  VS endpoint '{VS_ENDPOINT_NAME}' state: {state}")
            if state != "ONLINE":
                print("  [WARN] Endpoint not ONLINE — skipping sync")
                return
        else:
            print(f"  [WARN] Cannot check endpoint: HTTP {resp.status_code}")
            return
    except Exception as e:
        print(f"  [WARN] Cannot reach endpoint: {e}")
        return

    # Check if index exists; create if not
    idx_url = f"{ws_url}/api/2.0/vector-search/indexes/{VS_INDEX_NAME}"
    resp = requests.get(idx_url, headers=headers, timeout=30, verify=False)
    if resp.status_code == 404:
        print(f"  Creating VS index: {VS_INDEX_NAME}")
        create_body = {
            "name": VS_INDEX_NAME,
            "endpoint_name": VS_ENDPOINT_NAME,
            "primary_key": "chunk_id",
            "index_type": "DELTA_SYNC",
            "delta_sync_index_spec": {
                "source_table": CHUNK_TABLE,
                "pipeline_type": "TRIGGERED",
                "embedding_vector_columns": [
                    {
                        "name": "embedding",
                        "embedding_dimension": EMBEDDING_DIMENSION,
                    }
                ],
            },
        }
        create_resp = requests.post(
            f"{ws_url}/api/2.0/vector-search/indexes",
            headers=headers, json=create_body, timeout=60, verify=False,
        )
        if create_resp.ok:
            print(f"  [OK] VS index created: {VS_INDEX_NAME}")
        else:
            print(f"  [WARN] VS index create failed: HTTP {create_resp.status_code} "
                  f"{create_resp.text[:300]}")
            return
    elif not resp.ok:
        print(f"  [WARN] Cannot check index: HTTP {resp.status_code}")
        return
    else:
        print(f"  [OK] VS index exists: {VS_INDEX_NAME}")

    # Trigger sync
    sync_url = f"{ws_url}/api/2.0/vector-search/indexes/{VS_INDEX_NAME}/sync"
    for attempt in range(1, 4):
        try:
            resp = requests.post(sync_url, headers=headers, json={},
                                 timeout=30, verify=False)
            if resp.ok:
                print(f"  [OK] VS sync triggered")
                return
            if resp.status_code == 400 and "not ready" in resp.text.lower():
                print(f"  Endpoint warming (attempt {attempt}/3) — waiting 20s...")
                time.sleep(20)
                continue
            print(f"  [WARN] VS sync failed: HTTP {resp.status_code} {resp.text[:200]}")
            return
        except Exception as e:
            print(f"  [WARN] VS sync error: {e}")
            return

    print("  [WARN] VS sync failed after 3 attempts")


print("--- Vector Search Sync ---")
sync_vector_search()

# ── Final summary ───────────────────────────────────────────────────────────
print("\n--- Final metadata table state ---")
display(spark.sql(f"""
    SELECT pdf_name, metadata_status, chunk_status, chunk_error
    FROM {METADATA_TABLE}
    ORDER BY pdf_name
"""))